# 02: Model Evaluation

Building a model is only half the job. Proper evaluation tells you:
- How good is your model?
- Which errors does it make?
- Is it suitable for production?

This notebook covers: confusion matrix, precision, recall, F1, ROC curves, and business trade-off analysis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score
)

## Part 1: Building a Classifier

We'll create a synthetic dataset simulating credit card fraud detection — an inherently imbalanced problem.

In [ ]:
# Create imbalanced dataset: 5% fraud, 95% legitimate
X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_classes=2,
    weights=[0.95, 0.05],  # 5% positive (fraud)
    flip_y=0.05,  # add some label noise
    random_state=42
)

print(f"Dataset shape: {X.shape}")
print(f"Class distribution: Legitimate={np.sum(y==0)}, Fraud={np.sum(y==1)}")
print(f"Fraud percentage: {np.mean(y)*100:.1f}%")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Train classifier
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

## Part 2: Confusion Matrix

The confusion matrix shows exactly what the model got right and wrong.

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(f"                 Predicted")
print(f"                 Neg    Pos")
print(f"Actual Neg    [ {cm[0,0]:4d}  | {cm[0,1]:4d} ]")
print(f"Actual Pos    [ {cm[1,0]:4d}  | {cm[1,1]:4d} ]")

tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives:  {tn}")
print(f"False Positives: {fp} (predicted fraud, but legitimate)")
print(f"False Negatives: {fn} (predicted legitimate, but fraud)")
print(f"True Positives:  {tp}")

In [ ]:
# Visual confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=["Legitimate", "Fraud"])
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix — Fraud Detection")
plt.tight_layout()
plt.show()

## Part 3: Precision, Recall, and F1

- **Precision**: Of all predicted fraud, how many were actually fraud?
- **Recall**: Of all actual fraud, how many did we catch?
- **F1**: Harmonic mean of precision and recall

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

print("\nFull classification report:")
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))

In [ ]:
# Why accuracy can be misleading with imbalanced data
# A dumb model that always predicts "Legitimate" would get:
dumb_accuracy = np.sum(y_test == 0) / len(y_test)
print(f"Dumb model (always predict Legitimate) accuracy: {dumb_accuracy:.4f}")
print(f"Our model accuracy:                             {accuracy:.4f}")
print(f"\nAccuracy is high because the dataset is imbalanced!")
print(f"Precision and recall tell the real story.")

## Part 4: ROC Curve and AUC

The ROC curve shows the trade-off between:
- **True Positive Rate (Recall)**: fraction of actual positives detected
- **False Positive Rate**: fraction of actual negatives incorrectly flagged

The AUC (Area Under Curve) summarizes performance across all thresholds.

In [ ]:
# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

print(f"AUC Score: {auc:.4f}")
print(f"\nAUC interpretation:")
print(f"  0.5 = random guessing")
print(f"  0.7 = acceptable")
print(f"  0.8 = good")
print(f"  0.9 = excellent")
print(f"  1.0 = perfect")

# Plot ROC curve
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color="#e74c3c", linewidth=2, label=f"Logistic Regression (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random Classifier")
ax.fill_between(fpr, tpr, alpha=0.1, color="#e74c3c")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate (Recall)")
ax.set_title("ROC Curve — Fraud Detection")
ax.legend(loc="lower right")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

In [ ]:
# Visualize threshold effect
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, tpr[:-1] if len(tpr) > len(thresholds) else tpr, label="Recall (TPR)", color="#2ecc71")
ax.plot(thresholds, fpr[:-1] if len(fpr) > len(thresholds) else fpr, label="FPR", color="#e74c3c")
ax.set_xlabel("Classification Threshold")
ax.set_ylabel("Rate")
ax.set_title("Effect of Threshold on Recall and FPR")
ax.legend()
ax.axvline(x=0.5, color="gray", linestyle="--", alpha=0.5, label="Default threshold")
plt.tight_layout()
plt.show()

## Part 5: Business Trade-off Analysis

Different use cases demand different precision/recall trade-offs.

### Scenario: Fraud Detection
- **High Recall** (catch more fraud): You might flag some legitimate transactions, but you catch more fraudsters. The cost of missing fraud is high (direct financial loss).
- **High Precision** (fewer false alarms): You flag fewer legitimate transactions as fraud, but you miss more actual fraud. Fewer angry customers.

### What should we optimize for?

In [ ]:
# Simulate different thresholds
thresholds_to_test = [0.3, 0.4, 0.5, 0.6, 0.7]
print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Caught/100 fraud':<18}")
print("-" * 66)

for thresh in thresholds_to_test:
    y_pred_thresh = (y_prob >= thresh).astype(int)
    p = precision_score(y_test, y_pred_thresh, zero_division=0)
    r = recall_score(y_test, y_pred_thresh)
    f = f1_score(y_test, y_pred_thresh)
    caught = r * 100
    print(f"{thresh:<12.1f} {p:<12.4f} {r:<12.4f} {f:<12.4f} {caught:<18.1f}")

print("\nLower threshold → higher recall, lower precision")
print("Higher threshold → higher precision, lower recall")

In [ ]:
# Cost analysis: what does each error type cost?
print("=== Cost Analysis: Fraud Detection ===")
print("Assume: average fraud = $500, investigation cost = $50")
print()

fraud_cost = 500
investigation_cost = 50

for thresh in [0.3, 0.5, 0.7]:
    y_pred_t = (y_prob >= thresh).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    
    # Cost of missed fraud (FN * fraud amount)
    missed_cost = fn_t * fraud_cost
    # Cost of investigating false alarms (FP * investigation cost)
    alarm_cost = fp_t * investigation_cost
    # Cost of catching fraud (savings)
    caught_savings = tp_t * fraud_cost
    
    total_cost = missed_cost + alarm_cost - caught_savings
    
    print(f"Threshold {thresh}:")
    print(f"  Missed fraud: {fn_t} transactions (${missed_cost:,})")
    print(f"  False alarms: {fp_t} investigations (${alarm_cost:,})")
    print(f"  Fraud caught: {tp_t} transactions (${caught_savings:,} saved)")
    print(f"  Net cost: ${total_cost:,}")
    print()

## Part 6: Comparing Models with Evaluation Metrics

Accuracy alone is not enough. Let's compare models across multiple metrics.

In [ ]:
# Compare Logistic Regression vs Random Forest
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
y_prob_rf = rf_clf.predict_proba(X_test)[:, 1]

# Compare metrics
metrics = {
    "Logistic Regression": {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
    },
    "Random Forest": {
        "Accuracy": accuracy_score(y_test, y_pred_rf),
        "Precision": precision_score(y_test, y_pred_rf),
        "Recall": recall_score(y_test, y_pred_rf),
        "F1": f1_score(y_test, y_pred_rf),
        "AUC": roc_auc_score(y_test, y_prob_rf),
    },
}

print(f"{'Metric':<15} {'Logistic Reg':<18} {'Random Forest':<18}")
print("-" * 51)
for metric_name in ["Accuracy", "Precision", "Recall", "F1", "AUC"]:
    lr_val = metrics["Logistic Regression"][metric_name]
    rf_val = metrics["Random Forest"][metric_name]
    winner = "←" if lr_val > rf_val else ("→" if rf_val > lr_val else "=")
    print(f"{metric_name:<15} {lr_val:<18.4f} {rf_val:<18.4f} {winner}")

In [ ]:
# ROC curves comparison
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_lr, tpr_lr, color="#3498db", linewidth=2,
        label=f"Logistic Regression (AUC={roc_auc_score(y_test, y_prob):.3f})")
ax.plot(fpr_rf, tpr_rf, color="#e74c3c", linewidth=2,
        label=f"Random Forest (AUC={roc_auc_score(y_test, y_prob_rf):.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve Comparison")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Accuracy is misleading** for imbalanced datasets — always check precision, recall, and F1.
2. **Confusion matrix** shows exactly what errors the model makes.
3. **Precision** measures how trustworthy positive predictions are.
4. **Recall** measures how many actual positives you catch.
5. **ROC/AUC** summarizes performance across all classification thresholds.
6. **Threshold tuning** lets you trade precision for recall based on business costs.
7. **Cost analysis** helps you pick the right threshold — optimize for business outcomes, not just metrics.

Next: [03-overfitting-underfitting.ipynb](03-overfitting-underfitting.ipynb) — diagnosing and fixing model problems.